# Blockpose detector — current status

Reproducible **seed 0** status of the geometry-only detector. Every 3D view below is
**interactive: drag to rotate, scroll to zoom, double-click to reset.**

Colour convention (matching `tools/view_scene.py`):

| | meaning |
|---|---|
| <span style="color:#00c853">**green wireframe**</span> | ground truth — *evaluation-only label* |
| <span style="color:#2979ff">**blue wireframe**</span> | detector output |
| <span style="color:#ff6d00">**orange diamond**</span> | sensor origin |

The detector never receives synthetic instance IDs, poses, or locations — only points and,
optionally, the sensor origin and ray directions.

**Where the two wireframes coincide, the detection is correct to well under the line width.**
After model refinement that is the expected appearance; a visible gap is a real error.

In [1]:
import sys
from pathlib import Path

import numpy as np


def find_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        candidate = candidate.resolve()
        if (candidate / "pyproject.toml").is_file() and (candidate / "python/blockpose").is_dir():
            return candidate
    raise RuntimeError("Could not locate repository root from notebook working directory")


REPO_ROOT = find_repo_root()
for path in (REPO_ROOT / "python", REPO_ROOT / "tools"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from blockpose.detection import (
    DetectionParams,
    SensorContext,
    detect_blocks_detailed,
    detect_scene_sample,
)
from blockpose.evaluate import evaluate
from blockpose.scenario_matrix import generate_scenario_matrix
from blockpose.synth import BlockSpec, SynthParams, generate_scene
from blockpose.viz import error_table, sample_figure
import eval_detector

SEED = 0
CASES = {case.name: sample for case, sample in generate_scenario_matrix(SEED)}
# Keep embedded figures small enough to version: ~8k markers renders smoothly and
# the notebook stays a couple of MB rather than tens.
MAX_POINTS = 6000
print(f"{len(CASES)} scenarios at seed {SEED}")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


28 scenarios at seed 0


In [2]:
def show(name, title=None, params=None, max_points=MAX_POINTS):
    """Detect a named scenario and return an interactive ground-truth overlay."""
    sample = CASES[name]
    detections = detect_scene_sample(sample, params or DetectionParams())
    result = evaluate(detections, sample.ground_truth)
    print(f"{name}: matched={result.matched}  FP={result.unmatched_det}  FN={result.unmatched_gt}")
    for row in error_table(detections, sample.ground_truth):
        if row["detection"] is None:
            print(f"   gt[{row['gt']}]  MISSED")
        else:
            print(f"   gt[{row['gt']}]  {row['translation_mm']:7.2f} mm   "
                  f"{row['rotation_deg']:5.2f} deg")
    return sample_figure(sample, detections, title=title or name, max_points=max_points)

## 1. A well-observed block

The nominal operating regime: one isolated block, top plus two side faces visible. Rotate the
view to see how little of the block the sensor actually gets — the far and bottom faces are
never observed, and the pose is still recovered to sub-centimetre accuracy.

In [3]:
show("isolated_tilt_7deg", "Isolated block, 7 deg tilt")

isolated_tilt_7deg: matched=1  FP=0  FN=0
   gt[0]     0.37 mm    0.02 deg


## 2. Multiple blocks with mutual occlusion

Three separated blocks. The nearer blocks shadow the farther ones; note the clean point
shadows on the ground behind each block when you rotate.

In [4]:
show("three_separated", "Three separated blocks")

three_separated: matched=3  FP=0  FN=0
   gt[0]     0.36 mm    0.08 deg
   gt[1]     0.28 mm    0.16 deg
   gt[2]     0.21 mm    0.06 deg


## 3. What model refinement changed

The closed-form pose came from a top plane plus at most one side plane, so yaw rested on a
single RANSAC normal. `blockpose/refine.py` now fits the exact cuboid to every assigned return
by minimising the analytic box SDF over the full 6-DOF pose.

Toggle `refine_enabled` to compare. The stacked case is the interesting one: its cluster holds
both blocks, so refinement only works because point assignment splits the returns first.

In [5]:
for name in ("isolated_noise_3cm", "offset_two_stack"):
    sample = CASES[name]
    print(f"--- {name}")
    for label, enabled in (("plane fit only ", False), ("+ SDF refinement", True)):
        detections = detect_scene_sample(sample, DetectionParams(refine_enabled=enabled))
        result = evaluate(detections, sample.ground_truth)
        translation = f"{1000 * max(result.translation_errors_m):7.2f} mm" if result.matched else "   n/a"
        rotation = f"{max(result.rotation_errors_deg):5.2f} deg" if result.matched else "  n/a"
        print(f"   {label}  worst: {translation}  {rotation}")

--- isolated_noise_3cm
   plane fit only   worst:   43.88 mm   1.87 deg


   + SDF refinement  worst:    4.73 mm   0.46 deg
--- offset_two_stack
   plane fit only   worst:   17.97 mm   4.39 deg


   + SDF refinement  worst:    3.52 mm   0.59 deg


In [6]:
show("offset_two_stack", "Offset two-block stack (refined)")

offset_two_stack: matched=2  FP=0  FN=0
   gt[0]     2.06 mm    0.59 deg
   gt[1]     3.52 mm    0.27 deg


## 4. Gate and stratified diagnostics

Only the isolated suite is gated. The scenario matrix is deliberately diagnostic: weak families
stay visible instead of being averaged into the headline number.

`recall`/`precision` are `None` where undefined — a hard-negative family has no ground truth
and correctly produces no detections, so `false_positives` is what carries the signal there.
`unobservable_gt` counts blocks that produced too few returns to be detectable at all; they are
excluded from the recall denominator rather than scored as misses.

In [7]:
metrics = eval_detector.run_suite(seed=SEED)
gate_keys = ("recall", "precision", "translation_p95_m", "rotation_p95_deg", "false_positives_empty")
print("GATE")
for key in gate_keys:
    print(f"   {key:24s} {metrics[key]}")

def fmt(value, spec=".3f"):
    return "  n/a" if value is None else format(value, spec)

print(f"\n{'family':18s} {'recall':>7} {'prec':>7} {'t_p95 m':>9} {'r_p95 deg':>10} {'FP':>4} {'unobs':>6}")
for family, values in metrics["diagnostics_stratified"].items():
    print(f"{family:18s} {fmt(values['recall'], '.2f'):>7} {fmt(values['precision'], '.2f'):>7} "
          f"{fmt(values['translation_p95_m'], '.4f'):>9} {fmt(values['rotation_p95_deg'], '.2f'):>10} "
          f"{values['false_positives']:>4} {values['unobservable_gt']:>6}")

GATE
   recall                   1.0
   precision                1.0
   translation_p95_m        0.005658134487233815
   rotation_p95_deg         0.21905638442652167
   false_positives_empty    0

family              recall    prec   t_p95 m  r_p95 deg   FP  unobs
isolated_range        1.00    1.00    0.0006       0.64    0      0
tilt                  1.00    1.00    0.0004       0.02    0      0
noise                 1.00    1.00    0.0047       0.46    0      0
dropout               1.00    1.00    0.0003       0.23    0      0
multi_block           1.00    1.00    0.0004       0.15    0      0
touching              0.00     n/a       n/a        n/a    0      0
stack                 0.83    1.00    0.0053       0.60    0      0
clutter               1.00    1.00    0.0004       0.17    0      0
hard_negative          n/a     n/a       n/a        n/a    0      0
standoff              0.50    1.00    0.0013       0.24    0      0
uneven_ground         0.80    1.00    0.0008       0.19

## 5. Named failures — inspect them yourself

These are the honest gaps. Rotate each view to see why the detector behaves as it does.

**Touching row (recall 0).** A flush row of three blocks has no measured internal interface and
is geometrically identical to one 1.8 m elongated solid. Note that
`elongated_full_height_negative` in the hard-negative family is *exactly that solid* — the two
scenes produce the same measurement. No geometric algorithm separates them; it is a question of
scene prior. The detector currently chooses precision and emits nothing.

**Two-block stack.** Only the observable top block is recovered. The buried interface supplies
no surface evidence, so the lower pose is not invented from the layout.

**This scales badly to the target application.** A wall is a row of blocks in contact, so it hits
the same missing-interface problem at working scale. On `config/scenes/wall.yaml` the detector
recovers **1 of 16 blocks**; the `assembly` family below pins the boundary precisely — blocks
with 10 cm gaps give full recall, blocks in contact give zero.

In [8]:
for name in ("spaced_row_10cm_gaps", "wall_course_single", "wall_two_courses"):
    sample = CASES[name]
    result = evaluate(detect_scene_sample(sample, DetectionParams()), sample.ground_truth)
    print(f"{name:24s} gt={len(sample.ground_truth):2d}  matched={result.matched:2d}  "
          f"FP={result.unmatched_det}")

spaced_row_10cm_gaps     gt= 4  matched= 4  FP=0
wall_course_single       gt= 5  matched= 0  FP=0


wall_two_courses         gt= 8  matched= 0  FP=0


In [9]:
show("wall_course_single", "Failure: wall course, blocks in contact")

wall_course_single: matched=0  FP=0  FN=5
   gt[0]  MISSED
   gt[1]  MISSED
   gt[2]  MISSED
   gt[3]  MISSED
   gt[4]  MISSED


In [10]:
show("touching_row", "Failure: seamless touching row — nothing emitted")

touching_row: matched=0  FP=0  FN=3
   gt[0]  MISSED
   gt[1]  MISSED
   gt[2]  MISSED


In [11]:
show("two_stack", "Partial: buried lower stack block not recovered")

two_stack: matched=1  FP=0  FN=1
   gt[1]     5.77 mm    0.60 deg
   gt[0]  MISSED


## 6. Standoff limit

Recall is 1.0 at 10 m and 0 by 25 m. The real snapshots contain returns to ~31 m, so this is a
live limitation. The cause is point starvation against fixed `cluster_min_size` / `min_inliers`,
not pose quality — errors stay small right up to the point where the block disappears entirely.

In [12]:
for distance in (10, 15, 20, 25):
    sample = CASES[f"standoff_{distance}m"]
    detections = detect_scene_sample(sample, DetectionParams())
    result = evaluate(detections, sample.ground_truth)
    visible = int(sample.visible_point_counts()[0])
    error = f"{1000 * result.translation_errors_m[0]:6.1f} mm" if result.matched else "   missed"
    print(f"{distance:2d} m   returns on block: {visible:5d}   {error}")

10 m   returns on block:   384      0.6 mm
15 m   returns on block:   175      1.3 mm


20 m   returns on block:   107      missed
25 m   returns on block:    66      missed


In [13]:
show("standoff_20m", "Standoff 20 m — point starvation")

standoff_20m: matched=0  FP=0  FN=1
   gt[0]  MISSED


## 7. Sensor model provenance

The generator's parameters are derived from the real snapshots by
`tools/measure_sensor_model.py`, not guessed.

| quantity | measured | generator default | verdict |
|---|---|---|---|
| range noise, near field | ~0.005 m | 0.010 m | ~2x pessimistic |
| range noise, 10–15 m | ~0.010–0.015 m | 0.010 m + slope | matched |
| incidence dependence | none detectable | isotropic | matched |
| areal angular pitch | 0.239° | 0.25° | matched within 5% |
| nearest-neighbour gap | 0.133° | — | *within-scanline only* |

The two angular figures differ by ~2x because the scan pattern is non-repetitive: dense along
scan lines, sparse across them. The **areal** figure governs how many points land on a block
face. Matching the nearest-neighbour gap instead would overstate on-face density by ~3.5x and
make the whole suite optimistic.

## 8. Validity calibration — still deferred

Confidence estimates whether a returned detection is valid. It is separate from pose
observability and must not filter detections until calibrated. The selected-output synthetic
corpus still contains too few organic false positives to fit or evaluate a calibrator honestly,
so `calibrated_confidence` remains `None`.

This is the mechanism that would resolve the touching-row ambiguity properly: emit the tiling
hypothesis at low confidence and let the planner decide, rather than staying silent.

In [14]:
sample = CASES["isolated_near"]
context = SensorContext(sample.sensor_origin, sample.ray_directions, sample.ranges,
                        sample.miss_directions)
result = detect_blocks_detailed(sample.points, sensor_context=context)
for diagnostic in result.diagnostics:
    print(f"observability = {diagnostic.observability.value}")
    print(f"calibrated_confidence = {diagnostic.calibrated_confidence}")
    for name, value in diagnostic.features.items():
        print(f"   {name:24s} {value:.3f}")

observability = full
calibrated_confidence = None
   support                  0.835
   height_consistency       0.970
   visible_face_coverage    1.000
   free_space_consistency   1.000
   surface_explained        0.814


## 9. Known gaps

| gap | status |
|---|---|
| **Blocks in contact emit nothing** | the target application: a wall course scores 0, a 10 cm-gap row scores 1.0. Genuinely ambiguous vs the elongated hard negative, so it needs scored hypotheses and a scene prior, not better geometry |
| Standoff cliff at 15–20 m | fixed thresholds, not information — a face still has ~70 returns at 20 m |
| Structured dropout fragments DBSCAN | fixed-eps density clustering splits a striped cluster into sub-threshold pieces |
| Hypothesis generation not reproducible | Open3D 0.19 `segment_plane` is not seedable; refined *poses* are stable but candidate *counts* vary |
| Confidence calibration unwired | infrastructure exists and is tested, but nothing calls it |
| Real-data validation | blocked on per-snapshot TF re-export |
| No latency measurement | ~110 ms/scene measured ad hoc; not tracked by the gate |